# Customer Dataset Eda
#### 1. Missing Values
#### 2. Explore About The Numerical Vriable
#### 3. Explore Categorical variable(how may categories are there)
#### 4. Finding Relationships between features

In [ ]:
!pip install seaborn
import pandas as pandas
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

In [ ]:
pandas.set_option("display.max_columns", None)
customers = pandas.read_excel("../EABL dataset/CustomerMaster.xlsx", skiprows=3)
customers

In [ ]:
pandas.set_option("display.max_columns", None)
customers = pandas.read_excel("../EABL dataset/CustomerMaster.xlsx", skiprows=3)
customers

##  1. Missing Values

In [ ]:
customers.info()

In [ ]:
print(customers.shape)

In [ ]:
sns.heatmap(customers.isnull(),yticklabels=False,cbar=False,cmap='viridis')

In [ ]:
customers.isnull().sum()

### a)Drop Columns with 75% or more missing Values

In [ ]:
missing_pct = customers.isnull().mean()
cols_to_drop = missing_pct[missing_pct >= 0.75].index

print(f"Dropping {len(cols_to_drop)} columns with >=75% missing values:")
print(missing_pct[cols_to_drop].sort_values(ascending=False))

customers_clean = customers.drop(columns=cols_to_drop)
print(f"\nColumns before: {customers.shape[1]}, after: {customers_clean.shape[1]}")

In [ ]:
customers_clean.isnull().sum()

### b)Deal with BE Customer Ref No Missing values

In [ ]:
col = "BE Customer Ref No"  

null_mask = customers_clean[col].isna()
customers_clean.loc[null_mask]

In [ ]:
segment_cat = customers_clean["BE Customer Ref No"].astype("category")
codes = segment_cat.cat.codes  # each category mapped to an integer, alphabetical order by default

print("Mode:  ", customers_clean["BE Customer Ref No"].mode()[0])
print("Median:", codes.median(), f"-> category: {segment_cat.cat.categories[int(codes.median())]}")

In [ ]:
col = "BE Customer Ref No"  # change to whichever column

print(f"Number of unique values: {customers_clean[col].nunique()}")
print()

print("Unique values:")
print(customers_clean[col].unique())
print()

print("Mode (most frequent value):")
print(customers_clean[col].mode())
print()

print("Frequency of every unique value (mode is just the top row here):")
print(customers_clean[col].value_counts())


In [ ]:
missing_date_rows = customers_clean[customers_clean["BE Customer Ref No"].isna()]
print(f"Rows with missing BE Customer Ref No: {len(missing_date_rows)}")
print()

for col in [c for c in customers_clean.columns if c != "BE Customer Ref No"]:
    print(f"--- most common values of '{col}' among missing-BE Customer Ref No ---")
    print(missing_date_rows[col].value_counts().head(10))
    print()


In [ ]:
customers_clean = customers_clean.drop(columns=["BE Customer Ref No"])

### b)Deal with Customer Name2 Missing values

In [ ]:
customers_clean.isnull().sum()

In [ ]:
col = "Customer Name2"  

null_mask = customers_clean[col].notna()
customers_clean.loc[null_mask]

In [ ]:
col = "Customer Name2"  # change to whichever column

print(f"Number of unique values: {customers_clean[col].nunique()}")
print()

print("Unique values:")
print(customers_clean[col].unique())
print()

print("Mode (most frequent value):")
print(customers_clean[col].mode())
print()

print("Frequency of every unique value (mode is just the top row here):")
print(customers_clean[col].value_counts())

In [ ]:
import matplotlib.pyplot as plt

col = "Customer Name2"

# cardinality check first -- tells you whether a "distribution of categories" chart even makes sense here
n_total = len(customers_clean)
n_nonnull = customers_clean[col].notna().sum()
n_unique = customers_clean[col].nunique()

print(f"Total rows: {n_total}")
print(f"Non-null rows: {n_nonnull}")
print(f"Unique values: {n_unique}  ({n_unique / n_nonnull:.1%} of non-null rows are unique)")

if n_unique / n_nonnull > 0.5:
    print("\nWarning: most values here are unique -- this behaves more like a name/ID field "
          "than a true categorical variable. A bar chart of all categories would be unreadable "
          "(hundreds/thousands of bars each with count 1). Showing only the top 15 most repeated "
          "values below -- interpret this as 'names that happen to repeat,' not a real distribution.")

# --- plot: top 15 most frequent values, regardless of overall cardinality ---
top_values = customers_clean[col].value_counts().head(15).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_values.index, top_values.values, color="#3B6FA0", zorder=2)

ax.set_title(f"Most frequent values in {col} (top 15)")
ax.set_xlabel("Count")
ax.grid(axis="x", color="#e0e0e0", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
both_present = customers_clean[customers_clean["Customer Name"].notna() & customers_clean["Customer Name2"].notna()]
print(f"Rows where both Customer Name and Customer Name2 are non-null: {len(both_present)}")

# exact match
exact_match = both_present["Customer Name"] == both_present["Customer Name2"]
print(f"Exact match: {exact_match.sum()} ({exact_match.mean():.2%})")

# normalized match (case/whitespace-insensitive) -- exact string comparisons often miss near-duplicates
normalized_match = (
    both_present["Customer Name"].str.strip().str.upper()
    == both_present["Customer Name2"].str.strip().str.upper()
)
print(f"Match after normalizing case/whitespace: {normalized_match.sum()} ({normalized_match.mean():.2%})")

print()
print("Sample of rows where they DON'T match:")
print(both_present.loc[~normalized_match, ["Customer Name", "Customer Name2"]].head(15))


In [ ]:
name_norm = customers["Customer Name"].str.strip().str.upper()
name2_norm = customers["Customer Name2"].str.strip().str.upper()

both_null = customers["Customer Name"].isna() & customers["Customer Name2"].isna()
name2_null_only = customers["Customer Name"].notna() & customers["Customer Name2"].isna()
identical = customers["Customer Name2"].notna() & (name_norm == name2_norm)
different = customers["Customer Name2"].notna() & (name_norm != name2_norm)

print(f"Total rows: {len(customers)}")
print(f"Both null: {both_null.sum()}")
print(f"Customer Name2 null (Customer Name present): {name2_null_only.sum()}")
print(f"Identical values (normalized): {identical.sum()}")
print(f"Different values: {different.sum()}")
print(f"Check, should sum to {len(customers)}: {both_null.sum() + name2_null_only.sum() + identical.sum() + different.sum()}")


In [ ]:
customers_clean["has_alternate_name"] = customers_clean["Customer Name2"].notna()
customers_clean = customers_clean.drop(columns=["Customer Name2"])


### c)Deal with Price Group Missing Values

In [ ]:
customers_clean.isnull().sum()

In [ ]:
customers_clean

In [ ]:
col2 = "Price Group"  

null_mask = customers_clean[col2].isna()
customers_clean.loc[null_mask]

In [ ]:
customers_clean.iloc[10:20]

In [ ]:
col2 = "Privacy Policy"  

null_mask = customers_clean[col2].notna()
customers_clean.loc[null_mask]

In [ ]:
col = "Price Group"  # change to whichever column

print(f"Number of unique values: {customers_clean[col].nunique()}")
print()

print("Unique values:")
print(customers_clean[col].unique())
print()

print("Mode (most frequent value):")
print(customers_clean[col].mode())
print()

print("Frequency of every unique value (mode is just the top row here):")
print(customers_clean[col].value_counts())

In [ ]:
import matplotlib.pyplot as plt

col = "Price Group"

# cardinality check first -- tells you whether a "distribution of categories" chart even makes sense here
n_total = len(customers_clean)
n_nonnull = customers_clean[col].notna().sum()
n_unique = customers_clean[col].nunique()

print(f"Total rows: {n_total}")
print(f"Non-null rows: {n_nonnull}")
print(f"Unique values: {n_unique}  ({n_unique / n_nonnull:.1%} of non-null rows are unique)")

if n_unique / n_nonnull > 0.5:
    print("\nWarning: most values here are unique -- this behaves more like a name/ID field "
          "than a true categorical variable. A bar chart of all categories would be unreadable "
          "(hundreds/thousands of bars each with count 1). Showing only the top 15 most repeated "
          "values below -- interpret this as 'names that happen to repeat,' not a real distribution.")

# --- plot: top 15 most frequent values, regardless of overall cardinality ---
top_values = customers_clean[col].value_counts().head(15).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_values.index, top_values.values, color="#3B6FA0", zorder=2)

ax.set_title(f"Most frequent values in {col} (top 15)")
ax.set_xlabel("Count")
ax.grid(axis="x", color="#e0e0e0", linewidth=0.8, zorder=0)
ax.set_axisbelow(True)
for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
missing_date_rows = customers_clean[customers_clean["Price Group"].isna()]
print(f"Rows with missing visit_date: {len(missing_date_rows)}")
print()

for col in [c for c in customers_clean.columns if c != "Price Group"]:
    print(f"--- most common values of '{col}' among missing-date rows ---")
    print(missing_date_rows[col].value_counts().head(10))
    print()

In [ ]:
import pandas as pd
pd.crosstab(customers_clean["Segment"], customers_clean["Price Group"])

In [ ]:
col2 = "Price Group"  

null_mask = customers_clean[col2].isna()
customers_clean.loc[null_mask]

### Since Price Group is a categrory created by the buisness imputing uassigned is import because:
#### i)Price group is not a statisitcal data derived from nature but assignmed by the company thus hard to predict.
#### ii) Group based imputation could not work becasue only 78% being the average of all the right guess(Using different groups of the secgemt column to determine if a segment from the segment column  can help us determin what is the mode of the price group in that segment category)where mode is the right guess.

In [ ]:

pg = 'Price Group'

customers_clean['price_group_missing'] = customers_clean[pg].isna()

customers_clean[pg] = customers_clean[pg].fillna('Unassigned')

# sanity check
print(customers_clean['price_group_missing'].sum(), 'rows flagged as missing')
print(customers_clean[pg].value_counts(dropna=False))


## d)Deal with Price Group Missing Values

In [ ]:
customers_clean.isna().sum()

In [ ]:
col2 = "Outstanding Balance"  

null_mask = customers_clean[col2].notna()
customers_clean.loc[null_mask]

In [ ]:
customers_clean["Outstanding Balance"] = customers_clean["Outstanding Balance"].fillna(0)


In [ ]:
customers_clean.isna().sum()

In [ ]:
col2 = 'GPS Co-ordinate'

null_mask = customers_clean[col2].isna()
customers_clean.loc[null_mask]

### e)Dealing with RSS Date Registed missing values

In [ ]:
customers_clean.isna().sum()

In [ ]:
col2 = 'RSS Date Registed'

null_mask = customers_clean[col2].isna()
customers_clean.loc[null_mask]

In [ ]:
# Displays index 6 through 19 (Rows 7 to 20 based on positional indexing)
print(customers_clean[col2].iloc[65:100])


In [ ]:
col = 'RSS Date Registed'
needs_date = (customers_clean['RSS Flag'] == 'Yes') & customers_clean[col].isna()

ffilled = customers_clean[col].ffill()
bfilled = customers_clean[col].bfill()
agree = needs_date & (ffilled == bfilled)

customers_clean['rss_date_filled'] = False
customers_clean.loc[agree, col] = ffilled[agree]
customers_clean.loc[agree, 'rss_date_filled'] = True

# Rows sitting at a batch boundary (ffill/bfill disagree) stay unresolved
customers_clean['rss_date_unresolved'] = needs_date & ~agree

print(agree.sum(), 'rows filled (ffill and bfill agreed)')
print(customers_clean['rss_date_unresolved'].sum(), 'rows left unresolved (ffill and bfill disagreed)')


In [ ]:
print(customers_clean[col].iloc[65:100])

In [ ]:
customers_clean.isna().sum()

In [ ]:
col2 = 'RSS Date Registed'

null_mask = customers_clean[col2].isna()
customers_clean.loc[null_mask]

In [ ]:
customers_clean

In [ ]:
print(customers_clean['RSS Date Registed'].iloc[450:500])

In [ ]:

col = 'RSS Date Registed'

# Step 1 (from before): safely fill the 18 rows where ffill/bfill agreed
needs_date = (customers_clean['RSS Flag'] == 'Yes') & customers_clean[col].isna()
ffilled = customers_clean[col].ffill()
bfilled = customers_clean[col].bfill()
agree = needs_date & (ffilled == bfilled)
customers_clean.loc[agree, col] = ffilled[agree]

# Step 2: parse the raw strings into real datetimes, then collapse each
# distinct timestamp into a batch label, ordered chronologically
parsed = pd.to_datetime(customers_clean[col], format='%d/%m/%Y %I:%M:%S %p', errors='coerce')
unique_batches = sorted(parsed.dropna().unique())
batch_label = {
    ts: f'Batch_{i+1:03d}_{pd.Timestamp(ts).date()}'
    for i, ts in enumerate(unique_batches)
}
customers_clean['registration_batch'] = parsed.map(batch_label)

# Step 3: overwrite with honest categories for the two special cases,
# instead of leaving them as an unlabeled NaN batch
customers_clean.loc[customers_clean['RSS Flag'] == 'No', 'registration_batch'] = 'Not Registered'
still_unresolved = (customers_clean['RSS Flag'] == 'Yes') & customers_clean[col].isna()
customers_clean.loc[still_unresolved, 'registration_batch'] = 'Unresolved'

print(customers_clean['registration_batch'].value_counts(dropna=False).head(10))


In [ ]:
customers_clean.isna().sum()

### f)Dealing with NaN RSS First Login Date and RSS Last Login Date 

In [ ]:
col2 = 'RSS First Login Date'

null_mask = customers_clean[col2].isna()
customers_clean.loc[null_mask]

In [ ]:

customers_clean = customers_clean.dropna(axis=1, how='all')  # drop fully-empty unnamed columns

date_fmt = '%d/%m/%Y %I:%M:%S %p'
reg   = pd.to_datetime(customers_clean['RSS Date Registed'], format=date_fmt, errors='coerce')
first = pd.to_datetime(customers_clean['RSS First Login Date'], format=date_fmt, errors='coerce')
last  = pd.to_datetime(customers_clean['RSS Last Login Date'], format=date_fmt, errors='coerce')

# Use the latest date seen anywhere in the RSS columns as "today" for this
# dataset, rather than the real current date — safer since we don't know
# exactly when this file was extracted.
as_of_date = max(reg.max(), first.max(), last.max())

# #19 — missing indicator
customers_clean['ever_logged_in'] = first.notna()

# #20 / #35 — meaningful category instead of a bare NaN
customers_clean['login_status'] = 'Registered — Never Logged In'
customers_clean.loc[customers_clean['RSS Flag'] == 'No', 'login_status'] = 'Never Registered'
customers_clean.loc[customers_clean['ever_logged_in'], 'login_status'] = 'Logged In'

# #28 — the raw date columns are left untouched (still NaN where unknown)

# #30 — a more useful derived feature; NaN correctly stays NaN for
# customers who never logged in (nothing to compute a recency from)
customers_clean['days_since_last_login'] = (as_of_date - last).dt.days

print(customers_clean['login_status'].value_counts())

# ---------------------------------------------------------------
# #27 — survival-analysis-ready table (separate from the fill above)
# Question: "how long until a registered customer's first login?"
# Only includes customers with a known registration date.
# event_observed = 1 -> we saw them log in (duration = registration -> first login)
# event_observed = 0 -> censored: hasn't logged in yet as of as_of_date
#                        (duration = registration -> as_of_date, a lower bound)
# ---------------------------------------------------------------
survival_df = customers_clean.loc[reg.notna(), ['Customer Name']].copy()

logged_in = reg.notna() & first.notna()
censored  = reg.notna() & first.isna()

survival_df.loc[logged_in, 'duration_days'] = (first[logged_in] - reg[logged_in]).dt.days
survival_df.loc[censored, 'duration_days']  = (as_of_date - reg[censored]).dt.days
survival_df['event_observed'] = logged_in.loc[survival_df.index].astype(int)

print(survival_df.head())


In [ ]:
customers_clean.isna().sum()

In [ ]:
col = 'RSS Date Registed'
needs_date = (customers_clean['RSS Flag'] == 'Yes') & customers_clean[col].isna()

ffilled = customers_clean[col].ffill()
bfilled = customers_clean[col].bfill()
agree = needs_date & (ffilled == bfilled)

customers_clean['rss_date_filled'] = False
customers_clean.loc[agree, col] = ffilled[agree]
customers_clean.loc[agree, 'rss_date_filled'] = True

# Rows sitting at a batch boundary (ffill/bfill disagree) stay unresolved
customers_clean['rss_date_unresolved'] = needs_date & ~agree

print(agree.sum(), 'rows filled (ffill and bfill agreed)')
print(customers_clean['rss_date_unresolved'].sum(), 'rows left unresolved (ffill and bfill disagreed)')

In [ ]:
customers_clean.isna().sum()

### g)Deal with GPS Co-ordinate missing values

In [ ]:
customers_clean.isna().sum()

In [ ]:
col = 'GPS Co-ordinate'

customers_clean['gps_missing'] = customers_clean[col].isna()

print(customers_clean['gps_missing'].sum(), 'rows flagged as missing GPS')
print(customers_clean[col].isna().sum(), 'remain NaN (unchanged)')


In [ ]:
col2 = col

null_mask = customers_clean[col2].isna()
customers_clean.loc[null_mask]

In [ ]:
customers_clean.isna().sum()

In [ ]:
customers_clean

In [ ]:
# Second, separate flag — these are inferred from a matching Address string,
# not real GPS captures, so keep them distinguishable from real pins
customers_clean['gps_address_matched'] = False

known_gps = customers_clean.loc[~customers_clean[col].isna(), ['Address', col]].drop_duplicates(subset='Address', keep='first')
addr_to_gps = known_gps.set_index('Address')[col]

fillable = customers_clean['gps_missing'] & customers_clean['Address'].isin(addr_to_gps.index)
customers_clean.loc[fillable, col] = customers_clean.loc[fillable, 'Address'].map(addr_to_gps)
customers_clean.loc[fillable, 'gps_address_matched'] = True

print(fillable.sum(), 'rows filled via matching Address text')
print(customers_clean[col].isna().sum(), 'rows still genuinely missing GPS')


In [ ]:
customers_clean.isna().sum()

In [ ]:
col2 = col

null_mask = customers_clean[col2].isna()
customers_clean.loc[null_mask]

In [ ]:

col = 'GPS Co-ordinate'

# --- from before: flag + fill via matching Address text ---
customers_clean['gps_missing'] = customers_clean[col].isna()
customers_clean['gps_address_matched'] = False

known_gps = customers_clean.loc[~customers_clean[col].isna(), ['Address', col]].drop_duplicates(subset='Address', keep='first')
addr_to_gps = known_gps.set_index('Address')[col]

fillable = customers_clean['gps_missing'] & customers_clean['Address'].isin(addr_to_gps.index)
customers_clean.loc[fillable, col] = customers_clean.loc[fillable, 'Address'].map(addr_to_gps)
customers_clean.loc[fillable, 'gps_address_matched'] = True

customers_clean['gps_unresolved'] = customers_clean[col].isna()

still_missing = customers_clean[customers_clean['gps_unresolved']]
print(len(still_missing), 'rows still missing GPS after address-match fill')
print()
print('Cust Status breakdown:')
print(still_missing['Cust Status'].value_counts(dropna=False))
print()
print('Fallback geography available for these rows:')
for c in ['Territory', 'Area', 'Route']:
    print(f'  {c}: {still_missing[c].isna().sum()} missing / {len(still_missing)}')


In [ ]:
col2 = col

null_mask = customers_clean[col2].isna()
customers_clean.loc[null_mask]

# 2. Sales Data Explore About The Numerical Variable and  Outlier Treatment and 

In [ ]:
customers_clean[["Outstanding Balance","days_since_last_login"]].describe()

In [ ]:
customers_clean.isna().sum()

In [ ]:

# OR save as Excel, if you'd rather keep it in that format
customers_clean.to_excel("../EABL dataset/CustomerMaster_cleanV1.xlsx", index=False)

print("Saved:", customers_clean.shape)


In [ ]:
cm = pd.read_excel("../EABL dataset/CustomerMaster_cleanV1.xlsx")
cm

In [ ]:
cm.isna().sum()